In [1]:
import os
from datetime import datetime #biblioteca para pegar o horario em tempo real do momento que utiliza
from openai import OpenAI
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
client = OpenAI()

In [2]:
SYSTEM_PROMPT = (
    """Você é um especialista em sistemas energéticos e analista de produtos da empresa GoodWe,
    focada em soluções de energia limpa e infraestrutura para carregamento de veículos elétricos.
    Seu objetivo é auxiliar no controle inteligente de energia de um ambiente comercial
    com múltiplos carregadores de veículos elétricos.
    O sistema deve: distribuir energia de forma inteligente, evitar sobrecarga da rede elétrica,
    priorizar estabilidade energética, integrar energia solar quando disponível,
    redistribuir potência dinamicamente, explicar decisões técnicas e sugerir otimizações.
    Variáveis importantes: consumo atual do estabelecimento, capacidade máxima da rede,
    quantidade de veículos, potência disponível, horário de pico, energia solar disponível,
    prioridade de carregamento. Responda de forma técnica, objetiva e clara."""
)


In [3]:
historico = [{"role": "system", "content": SYSTEM_PROMPT}]


In [4]:
hora_atual = datetime.now().hour
#horario de pico

if 18 <= hora_atual <= 21:
    pico = "sim"
    fator_reducao = 0.7
elif 8 <= hora_atual <= 17:
    pico = "moderado"
    fator_reducao = 0.9
else:
    pico = "nao"
    fator_reducao = 1.0

In [5]:
'''Função para calcular a distribuição do ambiente, seu limite, quantidade de carregadores, a geração solar do local e calculado com o fator redução
- O fator de potência (FP) é a relação entre a potência ativa (kW, trabalho útil) e a potência aparente (kVA, energia total fornecida) em um sistema de
corrente alternada. Um fator de potência baixo (abaixo de \(0,92\) no Brasil) indica ineficiência, resultando em excesso de energia reativa, que gera
multas na conta de luz e sobrecarga nos equipamentos. - '''

def calcular_distribuicao(consumo_ambiente, limite_rede, carregadores, geracao_solar, fator_reducao):
    potencia_disponivel = (limite_rede + geracao_solar - consumo_ambiente) * fator_reducao

    if potencia_disponivel <= 0:
        return {"status": "sobrecarga", "potencia_total": 0, "potencia_por_carregador": 0}

    if carregadores <= 0:
        return {"status": "erro", "potencia_total": potencia_disponivel, "potencia_por_carregador": 0}

    return {
        "status": "normal",
        "potencia_total": potencia_disponivel,
        "potencia_por_carregador": potencia_disponivel / carregadores,
    }

In [6]:
# Função para o chat da ia transmitir o resultado/analise

def chat_ia(mensagem_usuario):
    historico.append({"role": "user", "content": mensagem_usuario})

    resposta = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=historico,
        temperature=0.5,
    )

    texto = resposta.choices[0].message.content
    historico.append({"role": "assistant", "content": texto})
    return texto

In [7]:
# O que a IA precisa analisar

def analisar_ia(dados):
    prompt = (
        f"Analise o seguinte cenário energético:\n\n"
        f"Consumo do ambiente: {dados['consumo']} kW\n"
        f"Limite da rede: {dados['limite']} kW\n"
        f"Carregadores ativos: {dados['carregadores']}\n"
        f"Geração solar: {dados['solar']} kW\n"
        f"Horário atual: {dados['hora']}h\n"
        f"Situação de pico: {dados['pico']}\n"
        f"Potência total disponível: {dados['potencia_total']} kW\n"
        f"Potência por carregador: {dados['potencia_por_carregador']} kW\n\n"
        f"Explique:\n"
        f"- Se existe risco de sobrecarga\n"
        f"- Se a distribuição está eficiente\n"
        f"- O que pode ser otimizado\n"
        f"- Como a IA está ajudando o sistema"
    )
    return chat_ia(prompt)

In [8]:
#Tela inicial da IA

print("=" * 65)
print("GOODWE - SISTEMA INTELIGENTE DE ENERGIA")
print("=" * 65)
print("\nSistema iniciado com sucesso.\n")
print(f"Horário atual detectado: {hora_atual}h")

#É horário de pico?

if pico == "sim":
    print("Horário de pico detectado. Redução inteligente de potência ativada.")
elif pico == "moderado":
    print("Horário comercial detectado. Controle moderado de potência ativado.")
else:
    print("Fora do horário de pico. Potência máxima liberada.")

#Input do usuario, pode existir alterações para que aconteça automaticamente no sistema

consumo = float(input("\nInforme o consumo atual do ambiente (kW): "))
limite = float(input("Informe o limite máximo da rede elétrica (kW): "))
carregadores = int(input("Informe a quantidade de carregadores ativos: "))
solar = float(input("Informe a geração solar atual (kW): "))

#Resultado com da função
resultado = calcular_distribuicao(consumo, limite, carregadores, solar, fator_reducao)

print("\n" + "=" * 65)
print("ANÁLISE DO SISTEMA")
print("=" * 65)

#Alertas, erros e resultados finais
if resultado["status"] == "sobrecarga":
    print("\nALERTA DE SOBRECARGA")
    print("Não há potência disponível. Carregadores serão limitados automaticamente.")
elif resultado["status"] == "erro":
    print("\nERRO: Quantidade de carregadores inválida.")
else:
    print(f"\nPotência disponível: {resultado['potencia_total']:.2f} kW")
    print(f"Potência por carregador: {resultado['potencia_por_carregador']:.2f} kW")

    if resultado["potencia_por_carregador"] < 7:
        print("\nIA DETECTOU POSSÍVEL SOBRECARGA")
        print("Aplicando redução inteligente de potência.")
    else:
        print("\nSistema operando normalmente. Distribuição inteligente ativa.")

#Analisar se o usuario quer ver oq a IA tem a dizer
usar_ia = input("\nDeseja que a IA analise o sistema? (sim/nao): ").lower()

if usar_ia == "sim":
    print("\nAnalisando cenário energético...\n")

    dados = {
        "consumo": consumo,
        "limite": limite,
        "carregadores": carregadores,
        "solar": solar,
        "hora": hora_atual,
        "pico": pico,
        "potencia_total": resultado["potencia_total"],
        "potencia_por_carregador": resultado["potencia_por_carregador"],
    }

    resposta_ia = analisar_ia(dados)
    print("=" * 65)
    print("RESPOSTA DA IA")
    print("=" * 65)
    print("\n" + resposta_ia)

print("\n" + "=" * 65)
print("CHAT INTELIGENTE ATIVADO")
print("Digite 'sair' para encerrar.")
print("=" * 65)

while True:
    pergunta = input("\nVocê: ")

#final do código para encerrar o programa
    if pergunta.lower() == "sair":
        print("\nEncerrando sistema...")
        break

    resposta = chat_ia(pergunta)
    print(f"\nIA:\n{resposta}")

GOODWE - SISTEMA INTELIGENTE DE ENERGIA

Sistema iniciado com sucesso.

Horário atual detectado: 23h
Fora do horário de pico. Potência máxima liberada.

Informe o consumo atual do ambiente (kW): 50
Informe o limite máximo da rede elétrica (kW): 200
Informe a quantidade de carregadores ativos: 2
Informe a geração solar atual (kW): 200

ANÁLISE DO SISTEMA

Potência disponível: 350.00 kW
Potência por carregador: 175.00 kW

Sistema operando normalmente. Distribuição inteligente ativa.

Deseja que a IA analise o sistema? (sim/nao): sim

Analisando cenário energético...

RESPOSTA DA IA

Vamos analisar o cenário energético apresentado:

### 1. Risco de Sobrecarga
- **Consumo do ambiente:** 50.0 kW
- **Limite da rede:** 200.0 kW
- **Geração solar:** 200.0 kW
- **Potência total disponível:** 350.0 kW

Neste cenário, o consumo atual do ambiente (50.0 kW) está bem abaixo do limite da rede (200.0 kW). Portanto, não há risco de sobrecarga, pois a soma do consumo e da potência dos carregadores não u